#  Problem 1
## Borges and Daripa JCP Paper

### Problem
$$
\begin{align*}
u(x,y) = 3e^{x+y}(x-x^2)(y-y^2)+5\\
f(x,y) = 6e^{x+y}xy(-3+x+y+xy)
\end{align*}
$$

### Tables

Unit disk (r=1)

table 1, relative errors (infimum) for N vs M, dirichlet

table 2, fix N=64, infimum and l2 errors, varying M, trapezoidal and simpsons rule


# Imports

In [1]:
import numpy as np
import warnings
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation
import numpy as np
import time
import matplotlib.pyplot as plt
import pandas as pd

import os, sys

# Main project root
repo_root = r"C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research"
os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.append(repo_root)


from poisson_solver_testing import (
    generate_grid_values,
    generate_uniform_radial,
    generate_nonuniform_radial,
    generate_uniform_azimuthal,
    generate_fixed_nonuniform_azimuthal,
    generate_nonuniform_azimuthal,
    generate_cartesian_grid_on_disk,
    trap_2d_on_disk,
    compute_error_metrics,
    plot_on_disk_with_error,
    plot_on_disk,
    poisson_solver
)




# Problem Setup

In [2]:
# Problem Setup

# Radius
R = 1.0

# Radial mesh type: keep radial grid uniform here
rad_unif = 1

# Solution and data functions
u = lambda x, y: 3 * np.exp(x + y) * (x - x**2) * (y - y**2) + 5
f = lambda x, y: 6 * np.exp(x + y) * x * y * (-3 + x + y + x * y)

# --- NEW: Dirichlet and Neumann boundary data separately ---

# Dirichlet boundary data: u on the boundary
g_dirichlet = lambda x, y: u(x, y)

# Partial derivatives of u
def u_x(x, y):
    # u = 3 e^{x+y} (x - x^2)(y - y^2) + 5
    e = np.exp(x + y)
    A = (x - x**2) * (y - y**2)
    A_x = (1 - 2*x) * (y - y**2)
    return 3 * e * (A + A_x)  # derivative via product rule

def u_y(x, y):
    e = np.exp(x + y)
    A = (x - x**2) * (y - y**2)
    A_y = (x - x**2) * (1 - 2*y)
    return 3 * e * (A + A_y)

# Neumann boundary data: ∂u/∂n on r = 1 (n = (x, y) on unit circle)
g_neumann = lambda x, y: u_x(x, y) * x + u_y(x, y) * y

'''N_values = [64, 128, 256, 512, 1024, 2048]
M_values = [64, 128, 256, 512, 1024, 2048]
N_fixed = 64'''

N_values = [64, 128, 256, 512, 1024, 2048]
M_values = [64, 128, 256, 512, 1024, 2048]
N_fixed = 512

# ----------------------------------------------------------
# Methods to test
# ----------------------------------------------------------
methods = [
    dict(
        name="uniform_fft",
        label="Uniform Mesh",
        azu_unif=2,
        mesh_kind=None,
        use_nudft=None,
    ),
    dict(
        name="Fixed Nonuniform Mesh - NUDFT",
        label="Fixed Nonuniform Mesh - NUDFT",
        azu_unif=1,
        mesh_kind="rand",
        use_nudft=True,
    ),
    dict(
        name="Fixed Nonuniform Mesh - NUFFT",
        label="Fixed Nonuniform Mesh - NUFFT",
        azu_unif=1,
        mesh_kind="rand",
        use_nudft=False,
    ),
]

BC_MAP = {
    "dirichlet": 1,
    "neumann": 2,
}

QUAD_MAP = {
    "trapezoidal": 1,
    "simpson": 2,
}

# Helper Functions

In [3]:
ANGLE_MESH_CACHE = {}

def get_cached_angle_mesh(method_cfg, N, M):
    azu_unif = method_cfg["azu_unif"]
    mesh_kind = method_cfg["mesh_kind"]
    method_name = method_cfg["name"]

    if azu_unif == 2:
        return generate_uniform_azimuthal(N)

    if azu_unif == 1:
        key = (method_name, N, mesh_kind)
        if key not in ANGLE_MESH_CACHE:
            ANGLE_MESH_CACHE[key] = generate_fixed_nonuniform_azimuthal(
                N, kind=mesh_kind or "rand"
            )
        return ANGLE_MESH_CACHE[key]

    if azu_unif == 0:
        key = (method_name, N, M, mesh_kind)
        if key not in ANGLE_MESH_CACHE:
            ANGLE_MESH_CACHE[key] = generate_nonuniform_azimuthal(
                N, M, kind=mesh_kind or "rand"
            )
        return ANGLE_MESH_CACHE[key]

    raise ValueError("Incorrect index for 'azu_unif'")

def build_radial_mesh(M):
    if rad_unif == 1:
        return generate_uniform_radial(M, R)
    return generate_nonuniform_radial(M, R)

def run_single_case(N, M, method_cfg, bc_name, quad_name):
    bc_choice = BC_MAP[bc_name]
    quad_rule = QUAD_MAP[quad_name]

    azu_unif = method_cfg["azu_unif"]
    use_nudft = method_cfg["use_nudft"]

    iRadius = build_radial_mesh(M)
    iAngle  = get_cached_angle_mesh(method_cfg, N, M)

    x_coord, y_coord = generate_cartesian_grid_on_disk(iAngle, iRadius)

    # interior data and true solution
    f_values = generate_grid_values(f, x_coord, y_coord)
    u_true   = generate_grid_values(u, x_coord, y_coord)

    # --- CHANGED: boundary data depends on BC ---
    if bc_choice == 1:  # Dirichlet
        g_values = generate_grid_values(
            g_dirichlet, x_coord[:, M - 1], y_coord[:, M - 1]
        )
    elif bc_choice == 2:  # Neumann
        g_values = generate_grid_values(
            g_neumann, x_coord[:, M - 1], y_coord[:, M - 1]
        )
    else:
        raise ValueError("Unknown bc_choice")

    # n = 0 mode for Neumann (phi_0), empty for Dirichlet
    if bc_choice == 2:
        u_fourier_0 = u_true.mean(axis=0)
    else:
        u_fourier_0 = np.array([])

    t0 = time.perf_counter()
    try:
        u_approx = poisson_solver(
            f_values, g_values, u_fourier_0,
            N, M, iRadius, iAngle, R,
            quad_rule, bc_choice,
            rad_unif, azu_unif,
            use_nudft_angular=(use_nudft if use_nudft is not None else False),
            maxiter_nufft=50,
            tol_nufft=1e-8,
        )
        runtime = time.perf_counter() - t0

        _, linf_rel, _, l2_rel = compute_error_metrics(
            u_approx, u_true, iRadius, iAngle
        )

    except MemoryError:
        runtime = np.nan
        linf_rel = np.nan
        l2_rel = np.nan

    return {
        "method": method_cfg["name"],
        "label": method_cfg["label"],
        "N": N,
        "M": M,
        "bc": bc_name,
        "quad": quad_name,
        "runtime": runtime,
        "L_inf_rel": linf_rel,
        "L2_rel": l2_rel,
    }

# Run Code, table 1

In [4]:
table1_results = []

for method in methods:
    print(f"\n{method['label']} -- Table 1")
    for N in N_values:
        for M in M_values:
            res = run_single_case(
                N=N,
                M=M,
                method_cfg=method,
                bc_name="dirichlet",
                quad_name="trapezoidal",
            )
            table1_results.append(res)

            print(
                f"N={N:4d}, M={M:4d}, "
                f"L_inf_rel={res['L_inf_rel']:.3e}, "
                f"time={res['runtime']:.3f} s"
            )

df_table1 = pd.DataFrame(table1_results)


Uniform Mesh -- Table 1
N=  64, M=  64, L_inf_rel=4.548e-06, time=0.008 s
N=  64, M= 128, L_inf_rel=1.119e-06, time=0.003 s
N=  64, M= 256, L_inf_rel=2.777e-07, time=0.012 s
N=  64, M= 512, L_inf_rel=6.915e-08, time=0.013 s
N=  64, M=1024, L_inf_rel=1.725e-08, time=0.026 s
N=  64, M=2048, L_inf_rel=4.309e-09, time=0.067 s
N= 128, M=  64, L_inf_rel=4.548e-06, time=0.011 s
N= 128, M= 128, L_inf_rel=1.119e-06, time=0.008 s
N= 128, M= 256, L_inf_rel=2.777e-07, time=0.009 s
N= 128, M= 512, L_inf_rel=6.915e-08, time=0.025 s
N= 128, M=1024, L_inf_rel=1.725e-08, time=0.042 s
N= 128, M=2048, L_inf_rel=4.309e-09, time=0.105 s
N= 256, M=  64, L_inf_rel=4.548e-06, time=0.005 s
N= 256, M= 128, L_inf_rel=1.119e-06, time=0.008 s
N= 256, M= 256, L_inf_rel=2.777e-07, time=0.021 s
N= 256, M= 512, L_inf_rel=6.915e-08, time=0.049 s
N= 256, M=1024, L_inf_rel=1.725e-08, time=0.093 s
N= 256, M=2048, L_inf_rel=4.309e-09, time=0.208 s
N= 512, M=  64, L_inf_rel=4.548e-06, time=0.015 s
N= 512, M= 128, L_inf_rel

# Run code, table 2

In [5]:
table2_results = []

for method in methods:
    print(f"\n{method['label']} -- Table 2")
    for M in M_values:
        for quad_name in ["trapezoidal", "simpson"]:
            for bc_name in ["dirichlet", "neumann"]:
                res = run_single_case(
                    N=N_fixed,
                    M=M,
                    method_cfg=method,
                    bc_name=bc_name,
                    quad_name=quad_name,
                )
                table2_results.append(res)

                print(
                    f"M={M:4d}, quad={quad_name:12s}, bc={bc_name:9s}, "
                    f"L_inf_rel={res['L_inf_rel']:.3e}, "
                    f"L2_rel={res['L2_rel']:.3e}, "
                    f"time={res['runtime']:.3f} s"
                )

df_table2 = pd.DataFrame(table2_results)


Uniform Mesh -- Table 2
M=  64, quad=trapezoidal , bc=dirichlet, L_inf_rel=4.548e-06, L2_rel=2.661e-06, time=0.013 s
M=  64, quad=trapezoidal , bc=neumann  , L_inf_rel=1.236e-04, L2_rel=4.546e-05, time=0.014 s
M=  64, quad=simpson     , bc=dirichlet, L_inf_rel=7.489e-07, L2_rel=4.023e-07, time=0.120 s

C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]



M=  64, quad=simpson     , bc=neumann  , L_inf_rel=7.605e-07, L2_rel=2.045e-07, time=0.097 s
M= 128, quad=trapezoidal , bc=dirichlet, L_inf_rel=1.119e-06, L2_rel=6.548e-07, time=0.016 s
M= 128, quad=trapezoidal , bc=neumann  , L_inf_rel=3.040e-05, L2_rel=1.118e-05, time=0.019 s
M= 128, quad=simpson     , bc=dirichlet, L_inf_rel=9.518e-08, L2_rel=5.085e-08, time=0.213 s
M= 128, quad=simpson     , bc=neumann  , L_inf_rel=9.536e-08, L2_rel=2.567e-08, time=0.147 s
M= 256, quad=trapezoidal , bc=dirichlet, L_inf_rel=2.777e-07, L2_rel=1.624e-07, time=0.039 s
M= 256, quad=trapezoidal , bc=neumann  , L_inf_rel=7.541e-06, L2_rel=2.774e-06, time=0.040 s
M= 256, quad=simpson     , bc=dirichlet, L_inf_rel=1.199e-08, L2_rel=6.388e-09, time=0.257 s
M= 256, quad=simpson     , bc=neumann  , L_inf_rel=1.193e-08, L2_rel=3.216e-09, time=0.452 s
M= 512, quad=trapezoidal , bc=dirichlet, L_inf_rel=6.915e-08, L2_rel=4.045e-08, time=0.086 s
M= 512, quad=trapezoidal , bc=neumann  , L_inf_rel=1.878e-06, L2_rel=

C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]


M=  64, quad=simpson     , bc=dirichlet, L_inf_rel=8.984e+00, L2_rel=1.064e+00, time=0.215 s


...

M=1024, quad=simpson     , bc=neumann  , L_inf_rel=5.207e-02, L2_rel=1.866e-02, time=21.664 s
M=2048, quad=trapezoidal , bc=dirichlet, L_inf_rel=5.484e-01, L2_rel=2.227e-01, time=45.570 s
M=2048, quad=trapezoidal , bc=neumann  , L_inf_rel=5.207e-02, L2_rel=1.866e-02, time=44.201 s


C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]


M=2048, quad=simpson     , bc=dirichlet, L_inf_rel=5.484e-01, L2_rel=2.227e-01, time=51.007 s


C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]


M=2048, quad=simpson     , bc=neumann  , L_inf_rel=5.207e-02, L2_rel=1.866e-02, time=49.467 s


# View Results

In [6]:
def dash_if_nan(x):
    return "—" if pd.isna(x) else f"{x:.1e}"

for method in methods:
    label = method["label"]
    name = method["name"]

    print(f"\n{'='*80}")
    print(f"{label} : TABLE 1")
    print(f"{'='*80}")

    df1 = df_table1[df_table1["method"] == name].copy()
    pivot1 = df1.pivot(index="N", columns="M", values="L_inf_rel")
    pivot1 = pivot1.reindex(index=N_values, columns=M_values)
    display(pivot1.map(dash_if_nan))

    print(f"\n{'='*80}")
    print(f"{label} : TABLE 2")
    print(f"{'='*80}")

    df2 = df_table2[df_table2["method"] == name].copy()

    pivot2 = pd.concat(
        {
            ("Trapezoidal rule", "Dirichlet", "L_inf_rel"): df2[
                (df2["quad"] == "trapezoidal") & (df2["bc"] == "dirichlet")
            ].set_index("M")["L_inf_rel"],

            ("Trapezoidal rule", "Dirichlet", "L2_rel"): df2[
                (df2["quad"] == "trapezoidal") & (df2["bc"] == "dirichlet")
            ].set_index("M")["L2_rel"],

            ("Trapezoidal rule", "Neumann", "L_inf_rel"): df2[
                (df2["quad"] == "trapezoidal") & (df2["bc"] == "neumann")
            ].set_index("M")["L_inf_rel"],

            ("Trapezoidal rule", "Neumann", "L2_rel"): df2[
                (df2["quad"] == "trapezoidal") & (df2["bc"] == "neumann")
            ].set_index("M")["L2_rel"],

            ("Simpson's rule", "Dirichlet", "L_inf_rel"): df2[
                (df2["quad"] == "simpson") & (df2["bc"] == "dirichlet")
            ].set_index("M")["L_inf_rel"],

            ("Simpson's rule", "Dirichlet", "L2_rel"): df2[
                (df2["quad"] == "simpson") & (df2["bc"] == "dirichlet")
            ].set_index("M")["L2_rel"],

            ("Simpson's rule", "Neumann", "L_inf_rel"): df2[
                (df2["quad"] == "simpson") & (df2["bc"] == "neumann")
            ].set_index("M")["L_inf_rel"],

            ("Simpson's rule", "Neumann", "L2_rel"): df2[
                (df2["quad"] == "simpson") & (df2["bc"] == "neumann")
            ].set_index("M")["L2_rel"],
        },
        axis=1
    )

    pivot2 = pivot2.reindex(M_values)
    display(pivot2.map(dash_if_nan))


for method in methods:
    label = method["label"]
    name = method["name"]

    print(f"\n{label} runtimes -- Table 1")
    display(
        df_table1[df_table1["method"] == name][["N", "M", "runtime"]]
        .sort_values(["N", "M"])
        .reset_index(drop=True)
    )

    print(f"\n{label} runtimes -- Table 2")
    display(
        df_table2[df_table2["method"] == name][["M", "quad", "bc", "runtime"]]
        .sort_values(["M", "quad", "bc"])
        .reset_index(drop=True)
    )


Uniform Mesh : TABLE 1


M,64,128,256,512,1024,2048
N,,,,,,
64,4.5e-06,1.1e-06,2.8e-07,6.9e-08,1.7e-08,4.3e-09
128,4.5e-06,1.1e-06,2.8e-07,6.9e-08,1.7e-08,4.3e-09
256,4.5e-06,1.1e-06,2.8e-07,6.9e-08,1.7e-08,4.3e-09
512,4.5e-06,1.1e-06,2.8e-07,6.9e-08,1.7e-08,4.3e-09
1024,4.5e-06,1.1e-06,2.8e-07,6.9e-08,1.7e-08,4.3e-09
2048,4.5e-06,1.1e-06,2.8e-07,6.9e-08,1.7e-08,4.3e-09



Uniform Mesh : TABLE 2


Trapezoidal rule                             Simpson's rule           \
            Dirichlet            Neumann               Dirichlet            
            L_inf_rel   L2_rel L_inf_rel   L2_rel      L_inf_rel   L2_rel   
M                                                                           
64            4.5e-06  2.7e-06   1.2e-04  4.5e-05        7.5e-07  4.0e-07   
128           1.1e-06  6.5e-07   3.0e-05  1.1e-05        9.5e-08  5.1e-08   
256           2.8e-07  1.6e-07   7.5e-06  2.8e-06        1.2e-08  6.4e-09   
512           6.9e-08  4.0e-08   1.9e-06  6.9e-07        1.5e-09  8.0e-10   
1024          1.7e-08  1.0e-08   4.7e-07  1.7e-07        1.9e-10  1.0e-10   
2048          4.3e-09  2.5e-09   1.2e-07  4.3e-08        2.4e-11  1.3e-11   

                         
       Neumann           
     L_inf_rel   L2_rel  
M                        
64     7.6e-07  2.0e-07  
128    9.5e-08  2.6e-08  
256    1.2e-08  3.2e-09  
512    1.5e-09  4.0e-10  
1024   1.9e-10  5.0e-11  
2048   2.3e-11  6.3e-12


Fixed Nonuniform Mesh - NUDFT : TABLE 1


...

,M,quad,bc,runtime
0,64,simpson,dirichlet,0.215041
1,64,simpson,neumann,0.519177
2,64,trapezoidal,dirichlet,0.177438
3,64,trapezoidal,neumann,0.161479
4,128,simpson,dirichlet,0.536971
5,128,simpson,neumann,0.362940
6,128,trapezoidal,dirichlet,0.158803
7,128,trapezoidal,neumann,0.269064
8,256,simpson,dirichlet,0.596087
9,256,simpson,neumann,0.689608



Fixed Nonuniform Mesh - NUFFT runtimes -- Table 1


,N,M,runtime
0,64,64,1.054011
1,64,128,1.662079
2,64,256,3.582033
3,64,512,5.985384
4,64,1024,11.863936
5,64,2048,21.354122
6,128,64,1.130817
7,128,128,1.900237
8,128,256,3.367198
9,128,512,6.220521



Fixed Nonuniform Mesh - NUFFT runtimes -- Table 2


,M,quad,bc,runtime
0,64,simpson,dirichlet,1.817247
1,64,simpson,neumann,1.795370
2,64,trapezoidal,dirichlet,1.639051
3,64,trapezoidal,neumann,1.261937
4,128,simpson,dirichlet,1.781788
5,128,simpson,neumann,1.673455
6,128,trapezoidal,dirichlet,2.821248
7,128,trapezoidal,neumann,2.016738
8,256,simpson,dirichlet,4.322075
9,256,simpson,neumann,4.141253


# testing

In [7]:
# Extra test: use the "nonuniform" solvers on a UNIFORM angular grid for Table 2

def dash_if_nan(x):
    return "—" if pd.isna(x) else f"{x:.1e}"

# Copy your fixed-nonuniform NUDFT and NUFFT configs,
# but override azu_unif=2 so they use the uniform angular mesh iAngle.
test_methods_uniform_angles = [
    dict(
        name="Fake Nonuniform (NUDFT on uniform θ)",
        label="Fake Nonuniform (NUDFT on uniform θ)",
        azu_unif=2,          # force uniform angular grid
        mesh_kind=None,
        use_nudft=True,      # still go through NUDFT code path
    ),
    dict(
        name="Fake Nonuniform (NUFFT on uniform θ)",
        label="Fake Nonuniform (NUFFT on uniform θ)",
        azu_unif=2,          # force uniform angular grid
        mesh_kind=None,
        use_nudft=False,     # still go through NUFFT code path
    ),
]

# Re-run Table 2 for these test methods: N fixed, vary M
M_values = [64, 128, 256, 512, 1024, 2048]
N_fixed  = 512

table2_test_results = []
for method in test_methods_uniform_angles:
    print(f"\n{method['label']} -- Table 2 (uniform θ, N = {N_fixed})")
    for M in M_values:
        for quad_name in ["trapezoidal", "simpson"]:
            for bc_name in ["dirichlet", "neumann"]:
                res = run_single_case(
                    N=N_fixed,
                    M=M,
                    method_cfg=method,
                    bc_name=bc_name,
                    quad_name=quad_name,
                )
                table2_test_results.append(res)

df_table2_test = pd.DataFrame(table2_test_results)

# Print in the same format as your original Table 2
for method in test_methods_uniform_angles:
    label = method["label"]
    name  = method["name"]

    print(f"\n{'='*80}")
    print(f"{label} : TABLE 2 (Problem 1, uniform θ, N = {N_fixed})")
    print(f"{'='*80}")

    df2 = df_table2_test[df_table2_test["method"] == name].copy()

    cols = []

    # Trapezoidal, Dirichlet
    cols.append(("Trapezoidal rule", "Dirichlet", "||·||∞",
                 df2[(df2["quad"]=="trapezoidal") & (df2["bc"]=="dirichlet")]
                    .set_index("M")["L_inf_rel"]))
    cols.append(("Trapezoidal rule", "Dirichlet", "||·||2",
                 df2[(df2["quad"]=="trapezoidal") & (df2["bc"]=="dirichlet")]
                    .set_index("M")["L2_rel"]))

    # Trapezoidal, Neumann
    cols.append(("Trapezoidal rule", "Neumann", "||·||∞",
                 df2[(df2["quad"]=="trapezoidal") & (df2["bc"]=="neumann")]
                    .set_index("M")["L_inf_rel"]))
    cols.append(("Trapezoidal rule", "Neumann", "||·||2",
                 df2[(df2["quad"]=="trapezoidal") & (df2["bc"]=="neumann")]
                    .set_index("M")["L2_rel"]))

    # Simpson, Dirichlet
    cols.append(("Simpson's rule", "Dirichlet", "||·||∞",
                 df2[(df2["quad"]=="simpson") & (df2["bc"]=="dirichlet")]
                    .set_index("M")["L_inf_rel"]))
    cols.append(("Simpson's rule", "Dirichlet", "||·||2",
                 df2[(df2["quad"]=="simpson") & (df2["bc"]=="dirichlet")]
                    .set_index("M")["L2_rel"]))

    # Simpson, Neumann
    cols.append(("Simpson's rule", "Neumann", "||·||∞",
                 df2[(df2["quad"]=="simpson") & (df2["bc"]=="neumann")]
                    .set_index("M")["L_inf_rel"]))
    cols.append(("Simpson's rule", "Neumann", "||·||2",
                 df2[(df2["quad"]=="simpson") & (df2["bc"]=="neumann")]
                    .set_index("M")["L2_rel"]))

    data   = { (r,bc,norm): series for (r,bc,norm,series) in cols }
    pivot2 = pd.DataFrame(data)
    pivot2.index.name = "M"
    pivot2 = pivot2.reindex(M_values)
    pivot2 = pivot2.sort_index(axis=1, level=[0,1,2])

    display(pivot2.map(dash_if_nan))


Fake Nonuniform (NUDFT on uniform θ) -- Table 2 (uniform θ, N = 512)


C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]



Fake Nonuniform (NUFFT on uniform θ) -- Table 2 (uniform θ, N = 512)

Fake Nonuniform (NUDFT on uniform θ) : TABLE 2 (Problem 1, uniform θ, N = 512)


Simpson's rule                            Trapezoidal rule           \
          Dirichlet           Neumann                 Dirichlet            
             ||·||2   ||·||∞   ||·||2   ||·||∞           ||·||2   ||·||∞   
M                                                                          
64          4.0e-07  7.5e-07  2.0e-07  7.6e-07          2.7e-06  4.5e-06   
128         5.1e-08  9.5e-08  2.6e-08  9.5e-08          6.5e-07  1.1e-06   
256         6.4e-09  1.2e-08  3.2e-09  1.2e-08          1.6e-07  2.8e-07   
512         8.0e-10  1.5e-09  4.0e-10  1.5e-09          4.0e-08  6.9e-08   
1024        1.0e-10  1.9e-10  5.0e-11  1.9e-10          1.0e-08  1.7e-08   
2048        1.3e-11  2.4e-11  6.3e-12  2.3e-11          2.5e-09  4.3e-09   

                        
      Neumann           
       ||·||2   ||·||∞  
M                       
64    4.5e-05  1.2e-04  
128   1.1e-05  3.0e-05  
256   2.8e-06  7.5e-06  
512   6.9e-07  1.9e-06  
1024  1.7e-07  4.7e-07  
2048  4.3e-08  1.2e-07


Fake Nonuniform (NUFFT on uniform θ) : TABLE 2 (Problem 1, uniform θ, N = 512)


Simpson's rule                            Trapezoidal rule           \
          Dirichlet           Neumann                 Dirichlet            
             ||·||2   ||·||∞   ||·||2   ||·||∞           ||·||2   ||·||∞   
M                                                                          
64          4.0e-07  7.5e-07  2.0e-07  7.6e-07          2.7e-06  4.5e-06   
128         5.1e-08  9.5e-08  2.6e-08  9.5e-08          6.5e-07  1.1e-06   
256         6.4e-09  1.2e-08  3.2e-09  1.2e-08          1.6e-07  2.8e-07   
512         8.0e-10  1.5e-09  4.0e-10  1.5e-09          4.0e-08  6.9e-08   
1024        1.0e-10  1.9e-10  5.0e-11  1.9e-10          1.0e-08  1.7e-08   
2048        1.3e-11  2.4e-11  6.3e-12  2.3e-11          2.5e-09  4.3e-09   

                        
      Neumann           
       ||·||2   ||·||∞  
M                       
64    4.5e-05  1.2e-04  
128   1.1e-05  3.0e-05  
256   2.8e-06  7.5e-06  
512   6.9e-07  1.9e-06  
1024  1.7e-07  4.7e-07  
2048  4.3e-08  1.2e-07